# B2.12 · Securing the developers' coding agents

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B2.11 · Context engineering for the pipeline](https://spbreed.github.io/cyber-commons/lessons/B2.11.html)**.

| | |
|---|---|
| Tools used | Docker, Cilium, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Measure the default agent's blast radius and reachable credentials, then rank controls by friction.

**Why a security engineer needs it.** The IDE agent holds git credentials, cloud credentials and a shell, in an unmanaged environment. The control it builds is: the strongest containment a developer does not notice: credential deny-lists and workspace confinement first.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The coding agent on an engineer's laptop holds repository write access, a cloud credential, and whatever MCP servers were convenient. It is the highest-privilege agent in most organisations and the least governed.

> **At CyberTravels.** The Coding Agent on Alex's laptop holds repository write, a cloud credential and whatever MCP servers were convenient. It is the highest-privilege agent at CyberTravels and the least governed. R6, R7.

## 2 · The framework

```
   the highest-privilege agent in most organisations

   +--------------------------------------------+
   |  coding agent on an engineer's laptop      |
   |  repo:write . cloud creds . shell . MCP    |
   +--------------------------------------------+
       |            |             |
     no SSO     no egress     no telemetry
                 policy

   governed by: whatever the engineer clicked
```

The coding agent in a developer's IDE is the most privileged agent in most
organisations and the least governed. It sits upstream of everything this track
has built: it writes the code the pipeline later analyses.

What it holds by default:

- the developer's **git credentials** — push access to everything they can push to,
- the **whole monorepo** on disk, including files they never open,
- a **shell**, usually unrestricted,
- their **cloud credentials** in `~/.aws` or `~/.config/gcloud`,
- whatever **MCP servers** they connected, each with its own authority.

That is a production identity in an unmanaged environment, driven by a model
reading code from the internet.

The binding constraint here is not technical feasibility — it is **developer
tolerance**. A containment scheme that adds friction to the inner loop is
disabled within a week, and a disabled control protects nothing. So the design
goal is the strongest containment a developer does not notice.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">control</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">friction</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what developers actually experience</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">deny-list credential paths</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">0.0</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the agent cannot read ~/.aws, ~/.ssh, ~/.env. Developers never noticed.</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">workspace confinement</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">0.1</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the agent sees the open repo only. Occasionally annoying for monorepo hops.</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">egress allowlist</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">0.2</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">package registries and your VCS. Breaks the odd curl in a generated script.</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">gate &lt;code&gt;git_push&lt;/code&gt;</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">0.4</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">one confirmation before code leaves the machine. Noticed, usually accepted.</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">gate every write</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">0.9</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>confirmation per file write. Abandoned within a week, every time.</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">no shell at all</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">1.0</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>removes the inner loop. Nobody will use the agent.</b></td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The first four ship today and cost nothing anyone will complain about. The last two are the ones people propose in meetings, and they are uninstalled by Friday — a control that gets turned off is worth less than a weaker one that stays on.</div>

## 3 · The audit, as a skill an agent runs on itself

Everything in this lesson is a review someone has to remember to do. Written as a skill, it is a review that fires whenever an agent opens a repository — including this one.

The skill's central instruction is easy to miss and decides the outcome: **rate an injection finding by what the allowlist permits, not by the text of the injection.** The payload is the attacker's choice and costs nothing to change; the allowlist is yours.

In [ ]:
# skills/appsec/coding-agent-hardening/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: coding-agent-hardening
description: >-
  Review the configuration of a coding agent — its skills, tools, hooks, MCP
  servers and autorun rules — for ways a repository can turn the agent against
  its operator. Use when asked to audit agent configuration, review a SKILL.md
  or AGENTS.md, assess MCP server risk, check tool allowlists, or secure
  developer AI tooling.
allowed-tools: Read, Grep, Glob
---

# Securing the developers' coding agents

A coding agent reads the repository it is working in. That makes every file in
the repository an input to a system that can run commands — and the agent's own
configuration is the control surface.

The threat is not that the agent is malicious. It is that instructions in a
repository are indistinguishable, at the token level, from instructions from
the operator.

## When to use this

Auditing `.claude/`, `AGENTS.md`, `.github/`, MCP configuration, CI that
invokes an agent, or any repository that a coding agent will open.

## Procedure

**1 — Inventory the surface.** List every file that reaches the agent's context
automatically: skill files, agent instruction files, hooks, MCP server
definitions, settings with tool allowlists, and any file the agent is told to
read on startup. Record which are **operator-controlled** (a maintainer wrote
them) and which are **content** (a contributor, a dependency, or a fetched
document can influence them).

**2 — Find the confusion.** For every content-controlled input, ask: if this
file contained an instruction, would the agent follow it? Grade each:

| Grade | Meaning |
|---|---|
| `isolated` | content is quoted as data and never read as instruction |
| `advisory` | content can influence phrasing but not tool calls |
| `directive` | content can cause a tool call |

Any `directive` path from unreviewed content is a finding, and its severity is
whatever the agent's most powerful allowed tool can do.

**3 — Check the allowlist against the blast radius.** For each pre-approved
tool, state the worst outcome of one call with attacker-chosen arguments. A
pre-approved `Bash` with unrestricted arguments is equivalent to pre-approving
everything else; note it as such rather than listing it as one item.

**4 — Check the hooks.** Hooks run without the model's judgement. A hook that
executes repository-supplied code (a script path from a config file, a test
command from `package.json`) runs attacker-supplied code the moment the agent
opens the repository. This is the highest-value finding in most reviews.

**5 — Check the MCP servers.** For each: who operates it, what it can reach,
whether its tool descriptions are trusted input (they are model-visible text
from a third party), and whether its credentials exceed the task.

**6 — Check the escape hatches.** Can a human stop it? Is there an audit trail
that survives the agent's own actions? An agent that can edit its own logs has
no logs.

## Output contract

```json
{
  "surface": [{"path": "str", "control": "operator|content", "auto_loaded": true}],
  "findings": [
    {"kind": "prompt_injection|overbroad_tool|unsafe_hook|mcp_trust|no_audit",
     "path": "str", "grade": "isolated|advisory|directive",
     "worst_case": "str", "severity": "critical|high|medium|low",
     "fix": "str"}
  ],
  "allowlist_review": [{"tool": "str", "worst_single_call": "str", "bounded": true}]
}
```

## Failure modes

- **Reviewing the agent's instructions and not its inputs.** The instructions
  are the part you control; the inputs are the part an attacker controls.
- **Treating tool descriptions as trusted.** They are third-party text placed
  directly in the model's context.
- **Assuming a human reviews every action.** Check whether the mode in use
  actually prompts, and audit the configuration that decides that.
- **Rating an injection finding by the text of the injection.** Rate it by what
  the allowlist permits.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/appsec/coding-agent-hardening/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/appsec/coding-agent-hardening/scripts/coding_agent_hardening.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Audit a coding agent's own configuration, tool list and MCP servers.

This is the executable half of the `coding-agent-hardening` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# The skill runtime comes from the shared library, not from a copy in this file.
# In a lesson notebook the cell above has already loaded it; standalone, find it
# the same way that cell does.
# The runtime comes from the shared library. The lesson cell above put it
# on the path; standalone, PYTHONPATH does (see scripts/test_skills.py).
from cyber_commons_skill_runtime import check, contract_of, parse_skill


def _skill_md():
    """The SKILL.md next to this script, or the one the notebook already parsed."""
    if "SKILL_MD" in globals():
        return globals()["SKILL_MD"]
    return (_pathlib.Path(__file__).resolve().parent.parent / "SKILL.md").read_text()


import pathlib as _pathlib

SKILL_MD = _skill_md()
meta, body = parse_skill(SKILL_MD)

SCOPE_WEIGHT = {"self":1,"project":3,"tenant":8,"org":20}
DEV_TOOLS = [
 # (name, writes, scope, reversible, needed in the inner loop)
 ("read_file",  False,"self",   True,  True),
 ("write_file", True, "project",True,  True),
 ("run_tests",  True, "self",   True,  True),
 ("run_shell",  True, "tenant", False, True),
 ("git_commit", True, "project",True,  True),
 ("git_push",   True, "project",False, False),
 ("read_env",   False,"org",    True,  False),
 ("http_get",   False,"self",   True,  True),
]
def blast(tools, gated=frozenset()):
    return sum(SCOPE_WEIGHT[s]*(1 if rev else 2)
               for n,w,s,rev,_ in tools if w and n not in gated)

print(f"{'tool':14s}{'writes':8s}{'scope':9s}{'reversible':12s}inner loop?")
print("-" * 58)
for n,w,s,rev,need in DEV_TOOLS:
    print(f"{n:14s}{str(w):8s}{s:9s}{str(rev):12s}{need}")
print(f"\ndefault blast radius: {blast(DEV_TOOLS)}")

import fnmatch
HOME = [
 "/home/dana/work/monorepo/src/app.py",
 "/home/dana/work/monorepo/.env",
 "/home/dana/work/other-team-repo/secrets.yaml",
 "/home/dana/.aws/credentials",
 "/home/dana/.ssh/id_ed25519",
 "/home/dana/.config/gcloud/application_default_credentials.json",
 "/home/dana/Downloads/customer-export-2026.csv",
]
def normalise(p):
    parts = []
    for seg in p.split("/"):
        if seg in ("", "."): continue
        if seg == "..":
            if parts: parts.pop()
            continue
        parts.append(seg)
    return "/" + "/".join(parts)

DENY = ("*/.ssh/*","*/.aws/*","*/.config/gcloud/*","*.pem","*/.env","*/Downloads/*")
def contained(p, workspace="/home/dana/work/monorepo"):
    real = normalise(p)
    if any(fnmatch.fnmatch(real, g) for g in DENY): return False
    return real.startswith(workspace + "/")

print(f"{'path':64s}{'default':9s}contained")
print("-" * 84)
for p in HOME:
    print(f"{p:64s}{'True':9s}{contained(p)}")
creds = [p for p in HOME if any(k in p for k in (".aws",".ssh","gcloud",".env"))]
print(f"\ncredential files reachable by default: {len(creds)}")
print(f"credential files reachable when contained: "
      f"{sum(1 for p in creds if contained(p))}")

gated = {"git_push"}
print(f"blast radius     {blast(DEV_TOOLS):>3} → {blast(DEV_TOOLS, gated):>3}")
print(f"reachable files  {len(HOME):>3} → {sum(1 for p in HOME if contained(p)):>3}")
print(f"credentials      {len(creds):>3} → {sum(1 for p in creds if contained(p)):>3}")
print(f"friction added   0.4 of 1.0 — one confirmation before a push")
assert not any(contained(p) for p in creds)
print("\nNo cloud or SSH credential is reachable, the inner loop is unchanged,")
print("and the only thing a developer notices is a prompt before pushing.")

contract = contract_of(body)

# What actually reaches a coding agent's context in a normal repository.
SURFACE = [
 ("AGENTS.md",             "operator", True),
 (".claude/settings.json", "operator", True),
 ("README.md",             "content",  True),
 ("docs/CONTRIBUTING.md",  "content",  True),
 ("package.json",          "content",  True),   # a hook may execute its scripts
 ("vendor/lib/README.md",  "content",  False),
]

worst = max(DEV_TOOLS, key=lambda t: SCOPE_WEIGHT[t[2]] * (1 if t[3] else 2))
audit = {
 "surface": [{"path": p, "control": c, "auto_loaded": a} for p, c, a in SURFACE],
 "findings": [
   {"kind": "prompt_injection", "path": "README.md", "grade": "directive",
    # severity is whatever the most powerful pre-approved tool can do
    "worst_case": f"content-controlled text triggers {worst[0]} at {worst[2]} scope",
    "severity": "critical" if not worst[3] else "high",
    "fix": "quote repository content as data; never as instruction"},
   {"kind": "unsafe_hook", "path": "package.json", "grade": "directive",
    "worst_case": "a hook runs a repo-supplied script the moment the repo opens",
    "severity": "critical",
    "fix": "pin the hook command; never take it from repository content"},
   {"kind": "overbroad_tool", "path": ".claude/settings.json", "grade": "advisory",
    "worst_case": f"{worst[0]} pre-approved with unrestricted arguments",
    "severity": "high", "fix": "split into narrow tools, or require approval"},
 ],
 "allowlist_review": [
   {"tool": name, "worst_single_call": f"{scope} scope, "
                                       f"{'reversible' if rev else 'IRREVERSIBLE'}",
    "bounded": scope in ("self", "project") and rev}
   for name, writes, scope, rev, _ in DEV_TOOLS if writes],
}
problems = check(audit, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

print(f"\nauto-loaded, content-controlled inputs: "
      f"{[s['path'] for s in audit['surface'] if s['control']=='content' and s['auto_loaded']]}")
print(f"most powerful pre-approved tool      : {worst[0]} ({worst[2]} scope)")
print("\nallowlist:")
for r in audit["allowlist_review"]:
    print(f"   {r['tool']:12s}{r['worst_single_call']:34s}bounded={r['bounded']}")
unbounded = [r["tool"] for r in audit["allowlist_review"] if not r["bounded"]]
print(f"\nunbounded pre-approved tools: {unbounded}")
print()
print("The injection finding is Critical not because the README says anything")
print("clever, but because a directive path exists to a tool that cannot be")
print("undone. Rewrite the payload and the severity does not move.")
assert unbounded, "an unbounded pre-approved tool should be visible here"

## What you just proved

The default developer agent scores a blast radius of 43 and can reach all seven paths including AWS, SSH and gcloud credentials. Containment reduces reachable paths to one source file with zero credentials reachable, and gating `git_push` drops the blast radius to 37 for 0.4 friction. The three lowest-friction controls remove every credential path without touching the inner loop.

## Your turn

Ship the credential deny-list first — it is a config file, it takes an afternoon, and no developer will notice. Then find out how many agents in your organisation could read `~/.aws/credentials` yesterday.

---

**Next → [B2.13 · Attesting control intent for agents and MCP servers](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*